# Telecom Customer Churn: Factor Identification and Prediction Using Machine Learning

**Project Title:** ტელეკომუნიკაციის მომხმარებელთა გადინების განმაპირობებელი ფაქტორების იდენტიფიცირება და პროგნოზირება მანქანური სწავლების მეთოდებით

**Author:** Mariam Tsintsadze  
**Supervisor:** Prof. Maxim Iavich  
**University:** Caucasus University, School of Technology  

---

## Table of Contents
1. [Import Libraries & Load Data](#1)
2. [Data Cleaning](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Statistical Testing](#4)
5. [Feature Engineering & Preprocessing](#5)
6. [Customer Segmentation](#6)
7. [Model Building (Without SMOTE - Baseline)](#7)
8. [Model Building (With SMOTE)](#8)
9. [Hyperparameter Tuning](#9)
10. [Model Evaluation & Comparison](#10)
11. [Precision-Recall Tradeoff & Threshold Optimization](#11)
12. [SHAP Analysis - Factor Identification](#12)
13. [LIME Validation](#13)
14. [Churn Risk Scoring System](#14)
15. [Business Recommendations](#15)
16. [Pipeline Summary](#16)
17. [Conclusion](#17)

---
<a id="1"></a>
## 1. Import Libraries & Load Data

In [1]:
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind

from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV,
    learning_curve
)

from sklearn.preprocessing import StandardScaler as SC
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier
)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    silhouette_score
)

from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap
import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

Dataset shape: (7043, 21)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Dataset Info:")
print("=" * 50)
df.info()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-n

In [4]:
print("\nStatistical Summary (Numerical):")
df.describe()


Statistical Summary (Numerical):


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
print("\nMissing Values:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")


Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicate Rows: 0


---
<a id="2"></a>
## 2. Data Cleaning

### 2.1 Data Type Verification

In [6]:
print("DATA TYPE VERIFICATION")
print("=" * 60)
print(f"{'Column':<20}{'Current Type':<15}{'Expected Type':<15}{'Status'}")
print("-" * 60)

expected_types = {
    'customerID': 'object', 'gender': 'object', 'SeniorCitizen': 'int64',
    'Partner': 'object', 'Dependents': 'object', 'tenure': 'int64',
    'PhoneService': 'object', 'MultipleLines': 'object', 'InternetService': 'object',
    'OnlineSecurity': 'object', 'OnlineBackup': 'object', 'DeviceProtection': 'object',
    'TechSupport': 'object', 'StreamingTV': 'object', 'StreamingMovies': 'object',
    'Contract': 'object', 'PaperlessBilling': 'object', 'PaymentMethod': 'object',
    'MonthlyCharges': 'float64', 'TotalCharges': 'float64', 'Churn': 'object'
}

issues_found = []
for col in df.columns:
    actual = str(df[col].dtype)
    expected = expected_types.get(col, 'unknown')
    if actual != expected:
        status = "Needs conversion"
        issues_found.append(col)
    else:
        status = "OK"
    print(f"{col:<20}{actual:<15}{expected:<15}{status}")

print("=" * 60)
print(f"\nIssues found: {len(issues_found)}")
if issues_found:
    print(f"  → {issues_found} — needs conversion")

DATA TYPE VERIFICATION
Column              Current Type   Expected Type  Status
------------------------------------------------------------
customerID          object         object         OK
gender              object         object         OK
SeniorCitizen       int64          int64          OK
Partner             object         object         OK
Dependents          object         object         OK
tenure              int64          int64          OK
PhoneService        object         object         OK
MultipleLines       object         object         OK
InternetService     object         object         OK
OnlineSecurity      object         object         OK
OnlineBackup        object         object         OK
DeviceProtection    object         object         OK
TechSupport         object         object         OK
StreamingTV         object         object         OK
StreamingMovies     object         object         OK
Contract            object         object         OK
PaperlessBi

### 2.2 Whitespace & Inconsistency Check in Categorical Columns

In [7]:
print("CATEGORICAL VALUE CONSISTENCY CHECK")
print("=" * 60)

cat_cols_check = df.select_dtypes(include='object').columns.tolist()
inconsistencies = 0

for col in cat_cols_check:
    unique_vals = df[col].unique()
    # Check for leading/trailing whitespace
    whitespace_issues = [v for v in unique_vals if isinstance(v, str) and (v != v.strip())]
    # Check for empty strings
    empty_issues = [v for v in unique_vals if isinstance(v, str) and v.strip() == '']
    
    if whitespace_issues or empty_issues:
        print(f"  ⚠ {col}: Found {len(whitespace_issues)} whitespace issues, {len(empty_issues)} empty values")
        inconsistencies += 1
    else:
        n_unique = len(unique_vals)
        print(f"  ✓ {col}: {n_unique} unique values — clean")

print("=" * 60)
if inconsistencies == 0:
    print("\n All categorical columns are clean — no whitespace or inconsistency issues.")
else:
    print(f"\n {inconsistencies} columns need cleaning.")

CATEGORICAL VALUE CONSISTENCY CHECK
  ✓ customerID: 7043 unique values — clean
  ✓ gender: 2 unique values — clean
  ✓ Partner: 2 unique values — clean
  ✓ Dependents: 2 unique values — clean
  ✓ PhoneService: 2 unique values — clean
  ✓ MultipleLines: 3 unique values — clean
  ✓ InternetService: 3 unique values — clean
  ✓ OnlineSecurity: 3 unique values — clean
  ✓ OnlineBackup: 3 unique values — clean
  ✓ DeviceProtection: 3 unique values — clean
  ✓ TechSupport: 3 unique values — clean
  ✓ StreamingTV: 3 unique values — clean
  ✓ StreamingMovies: 3 unique values — clean
  ✓ Contract: 3 unique values — clean
  ✓ PaperlessBilling: 2 unique values — clean
  ✓ PaymentMethod: 4 unique values — clean
  ⚠ TotalCharges: Found 1 whitespace issues, 1 empty values
  ✓ Churn: 2 unique values — clean

 1 columns need cleaning.


### 2.3 Missing Values Analysis

In [8]:
# Convert TotalCharges to numeric to reveal hidden missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("MISSING VALUES ANALYSIS")
print("=" * 60)
print(f"{'Column':<20}{'Missing':<10}{'% Missing':<12}{'Action'}")
print("-" * 60)

for col in df.columns:
    missing = df[col].isnull().sum()
    pct = missing / len(df) * 100
    if missing > 0:
        action = "Fill with 0 (new customers)"
        print(f"{col:<20}{missing:<10}{pct:<12.2f}{action}")
    
total_missing = df.isnull().sum().sum()
if total_missing == 0:
    print("  No missing values found in any column.")
else:
    print(f"\nTotal missing values: {total_missing}")
    print(f"Affected rows: {df.isnull().any(axis=1).sum()} ({df.isnull().any(axis=1).sum()/len(df)*100:.2f}%)")

print("=" * 60)

# Fix: Fill TotalCharges NaN with 0
print(f"\nCustomers with missing TotalCharges (tenure = 0, new customers):")
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges']].head(11))
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"\nMissing TotalCharges filled with 0. Remaining missing: {df.isnull().sum().sum()}")

MISSING VALUES ANALYSIS
Column              Missing   % Missing   Action
------------------------------------------------------------
TotalCharges        11        0.16        Fill with 0 (new customers)

Total missing values: 11
Affected rows: 11 (0.16%)

Customers with missing TotalCharges (tenure = 0, new customers):
      customerID  tenure  MonthlyCharges
488   4472-LVYGI       0           52.55
753   3115-CZMZD       0           20.25
936   5709-LVOEQ       0           80.85
1082  4367-NUYAO       0           25.75
1340  1371-DWPAZ       0           56.05
3331  7644-OMVMY       0           19.85
3826  3213-VVOLG       0           25.35
4380  2520-SGTTA       0           20.00
5218  2923-ARZLG       0           19.70
6670  4075-WKNIU       0           73.35
6754  2775-SEFEE       0           61.90

Missing TotalCharges filled with 0. Remaining missing: 0


### 2.4 Duplicate Records Check

In [9]:
duplicates = df.duplicated().sum()
duplicate_ids = df['customerID'].duplicated().sum()

print("DUPLICATE RECORDS CHECK")
print("=" * 40)
print(f"  Exact duplicate rows: {duplicates}")
print(f"  Duplicate customerIDs: {duplicate_ids}")

if duplicates == 0 and duplicate_ids == 0:
    print("\n No duplicates found — each row represents a unique customer.")
else:
    print(f"\n Found duplicates — removing...")
    df = df.drop_duplicates()

DUPLICATE RECORDS CHECK
  Exact duplicate rows: 0
  Duplicate customerIDs: 0

 No duplicates found — each row represents a unique customer.


### 2.5 Outlier Detection (IQR Method)

In [10]:
numerical_cols_outlier = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("OUTLIER ANALYSIS (IQR Method)")
print("=" * 70)
print(f"{'Feature':<18}{'Q1':<10}{'Q3':<10}{'IQR':<10}{'Lower':<10}{'Upper':<10}{'Outliers':<10}")
print("-" * 70)

for col in numerical_cols_outlier:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:<18}{Q1:<10.1f}{Q3:<10.1f}{IQR:<10.1f}{lower:<10.1f}{upper:<10.1f}{outliers:<10}")

print("=" * 70)
if outliers == 0:
    print("\n No outliers detected in numerical features.")
else:
    print(f"\nTotal outliers detected across all numerical features: {outliers}")

OUTLIER ANALYSIS (IQR Method)
Feature           Q1        Q3        IQR       Lower     Upper     Outliers  
----------------------------------------------------------------------
tenure            9.0       55.0      46.0      -60.0     124.0     0         
MonthlyCharges    35.5      89.8      54.3      -46.0     171.4     0         
TotalCharges      398.6     3786.6    3388.0    -4683.5   8868.7    0         

 No outliers detected in numerical features.


### 2.6 Data Cleaning Summary

DATA CLEANING SUMMARY

  1. Dataset shape: (7043, 21)
  2. Total missing values: 0 (fixed)
  3. Duplicates removed: 0
  4. Outliers: None found
  5. Type corrections: TotalCharges (object → float64)
  6. Whitespace issues: None found

✓ Dataset is clean and ready for feature engineering.